<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml20_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RNN(Recurrent Neural Network, 순환 신경망)

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras

In [2]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

# IMDB 데이터셋

*   imdb.com 사이트 사용자들의 영화 리뷰를 긍정(1), 부정(0)으로 분류한 데이터.
*   25,000개 훈련 샘플(영화 리뷰)와 25,000개의 테스트 샘플(영화 리뷰).
*   샘플들마다 사용된 단어(토큰, token) 개수가 다름.
    *   샘플들마다 특성(feature)의 개수가 다름 --> 전처리가 필요.
*   Keras datasets에서 다운로드한 데이터는 (영어) 단어들이 숫자로 인코딩된 상태.
    *   단어-숫자 dict
*   (참고) KoNLP 라이브러리 - 한국어 자연어 처리 라이브러리.

In [3]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=500)
# num_words=500: 가장 자주 사용된 500개의 단어만 인코딩. 그 이외의 단어들은 같은 숫자(500개 이외의 단어)로 인코딩.
# num_words=None: 기본값. 단어 사전에 포함된 모든 단어들을 인코딩.

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
x_train.shape  #> (25000,) 모양의 1차원 배열 - 25,000개의 영화 리뷰들.

(25000,)

In [5]:
x_test.shape

(25000,)

In [6]:
y_train.shape

(25000,)

In [7]:
np.unique(y_train, return_counts=True)  #> 훈련 레이블: 부정(0)/긍정(1) 리뷰는 같은 비율.

(array([0, 1]), array([12500, 12500]))

In [8]:
np.unique(y_test, return_counts=True)

(array([0, 1]), array([12500, 12500]))

## 훈련 셋 탐색

In [10]:
print(x_train[0])  # 첫번째 훈련 샘플(영화 리뷰)

[1, 14, 22, 16, 43, 2, 2, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 2, 112, 50, 2, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 2, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 2, 38, 76, 15, 13, 2, 4, 22, 17, 2, 17, 12, 16, 2, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 2, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 2, 36, 71, 43, 2, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]


In [11]:
print(type(x_train[0]))  #> list

<class 'list'>


In [12]:
len(x_train[0])  #> 첫번째 훈련 샘플의 길이(영화 리뷰에서 사용된 단어 개수)

218

In [13]:
print(x_train[1])  # 두번째 훈련 샘플(영화 리뷰)

[1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 2, 8, 118, 2, 14, 394, 20, 13, 119, 2, 189, 102, 5, 207, 110, 2, 21, 14, 69, 188, 8, 30, 23, 7, 4, 249, 126, 93, 4, 114, 9, 2, 2, 5, 2, 4, 116, 9, 35, 2, 4, 229, 9, 340, 2, 4, 118, 9, 4, 130, 2, 19, 4, 2, 5, 89, 29, 2, 46, 37, 4, 455, 9, 45, 43, 38, 2, 2, 398, 4, 2, 26, 2, 5, 163, 11, 2, 2, 4, 2, 9, 194, 2, 7, 2, 2, 349, 2, 148, 2, 2, 2, 15, 123, 125, 68, 2, 2, 15, 349, 165, 2, 98, 5, 4, 228, 9, 43, 2, 2, 15, 299, 120, 5, 120, 174, 11, 220, 175, 136, 50, 9, 2, 228, 2, 5, 2, 2, 245, 2, 5, 4, 2, 131, 152, 491, 18, 2, 32, 2, 2, 14, 9, 6, 371, 78, 22, 2, 64, 2, 9, 8, 168, 145, 23, 4, 2, 15, 16, 4, 2, 5, 28, 6, 52, 154, 462, 33, 89, 78, 285, 16, 145, 95]


In [14]:
len(x_train[1])

189

imdb 데이터셋(x_train, x_test)은 파이썬 list 객체들을 원소로 갖는 **1차원 ndarray**.

각각의 리스트들은 정수들을 저장하고 있음. 정수들은 영화 리뷰에서 사용된 단어(토큰)을 의미함.

각각의 리스트들은 길이가 다름. -> 2차원 ndarray로 만들 수 없음.

In [15]:
for i in range(5):
    print(f'인덱스 {i}: 토큰 개수 = {len(x_train[i])}')

인덱스 0: 토큰 개수 = 218
인덱스 1: 토큰 개수 = 189
인덱스 2: 토큰 개수 = 141
인덱스 3: 토큰 개수 = 550
인덱스 4: 토큰 개수 = 147


In [16]:
for i in range(5):
    print(x_train[i][:30])

[1, 14, 22, 16, 43, 2, 2, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 2, 112, 50, 2, 2, 9, 35, 480]
[1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 2, 8, 118, 2, 14, 394, 20, 13, 119, 2, 189, 102, 5, 207, 110]
[1, 14, 47, 8, 30, 31, 7, 4, 249, 108, 7, 4, 2, 54, 61, 369, 13, 71, 149, 14, 22, 112, 4, 2, 311, 12, 16, 2, 33, 75]
[1, 4, 2, 2, 33, 2, 4, 2, 432, 111, 153, 103, 4, 2, 13, 70, 131, 67, 11, 61, 2, 2, 35, 2, 2, 61, 2, 452, 2, 4]
[1, 249, 2, 7, 61, 113, 10, 10, 13, 2, 14, 20, 56, 33, 2, 18, 457, 88, 13, 2, 2, 45, 2, 13, 70, 79, 49, 2, 2, 13]


모든 리스트는 숫자 1로 시작. 숫자 1의 의미는 리뷰의 시작을 의미.